# 01 ViewRawData

Notebook นี้ใช้สำหรับดูข้อมูลเบื้องต้น

In [23]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR

sns.set_theme(style="whitegrid")
RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR

(WindowsPath('D:/ML/data/raw'),
 WindowsPath('D:/ML/data/processed'),
 WindowsPath('D:/ML/reports/figures'))

## การโหลดข้อมูล

โหลดไฟล์ข้อมูลจาก vwTimeStampDashbaord.csv

In [25]:
example_path = RAW_DATA_DIR / "vwTimeStampDashbaord.csv"
df = pd.read_csv(example_path)
df.head()

,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk-in,10/04/2026 10:01:58,9,1000000005,71-2054,SB1PL260410025,บ.ส.เอื้อสุข จก.,10/04/2026 10:02:03,N,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SB1,Walk-in,10/04/2026 09:58:49,8,1000000008,700-8513,SB1PL260410024,มบูเครือวัลย์ Lake and Park ลำผักชี,10/04/2026 09:59:06,N,...,34,NaN,NaN,NaN,NaN,10/04/2026 10:13:25,10/04/2026 10:20:44,10/04/2026 10:11:37,NaN,NaN
2,SB1,Walk-in,10/04/2026 09:57:56,7,1000000003,89-4045,SB1PL260410023,นครปฐม บางเลน,10/04/2026 09:57:58,N,...,0,NaN,NaN,NaN,NaN,10/04/2026 09:58:32,10/04/2026 09:59:48,NaN,NaN,NaN
3,SB1,Walk-in,10/04/2026 09:56:17,6,1000000005,80-8223,SB1PL260410022,บ.อุดมชัยกู๊ดโฮมสมุทรสงคราม จก.,10/04/2026 09:56:23,N,...,0,NaN,NaN,10/04/2026 10:19:38,10/04/2026 10:20:33,10/04/2026 10:00:25,10/04/2026 10:08:47,NaN,NaN,NaN
4,SB1,Walk-in,10/04/2026 09:48:27,5,1000000003,89-4045,SB1PL260410021,บ.จำหน่ายวัตถุก่อสร้าง จก.,10/04/2026 09:48:32,N,...,70,NaN,NaN,NaN,NaN,10/04/2026 09:49:36,NaN,10/04/2026 10:05:27,10/04/2026 10:10:23,NaN


## Data Type และขนาดข้อมูล

ดูจำนวนแถว, จำนวนคอลัมน์, และชนิดข้อมูลของแต่ละคอลัมน์

In [26]:
print(f"rows: {df.shape[0]:,}")
print(f"columns: {df.shape[1]:,}")

dtype_summary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
    }
)

dtype_summary.sort_values(["dtype", "column"]).reset_index(drop=True)

rows: 10,000
columns: 40


,column,dtype
0,Unnamed: 39,float64
1,ACCESSORIESSapAmount,int64
2,CPACFittingSapAmount,int64
3,CarType,int64
4,DURAFittingSapAmount,int64
5,NEUSTILEFittingSapAmount,int64
6,NEUSTILETileSapAmount,int64
7,PRESTIGEFittingSapAmount,int64
8,PRESTIGETileSapAmount,int64
9,TruckReceiveHour,int64


## Missing Values

สรุปคอลัมน์ที่มีค่าว่าง โดยนับรวมค่า `NULL` เป็น missing

In [27]:
null_like_tokens = ["NULL", "", "nan", "NaN", "None"]
df_profile = df.replace(null_like_tokens, np.nan)

missing_summary = pd.DataFrame(
    {
        "column": df_profile.columns,
        "missing_count": df_profile.isna().sum().values,
        "missing_pct": (df_profile.isna().mean() * 100).round(2).values,
    }
)

missing_summary = missing_summary.loc[missing_summary["missing_count"] > 0].copy()
missing_summary = missing_summary.sort_values("missing_count", ascending=False).reset_index(drop=True)
missing_summary

,column,missing_count,missing_pct
0,Unnamed: 39,10000,100.00
1,TruckOverTimeRemark,9964,99.64
2,TruckOverTimeName,9955,99.55
3,AccEnd,5175,51.75
4,AccStart,5154,51.54
5,FittingEnd,4252,42.52
6,FittingStart,4241,42.41
7,TileEnd,1831,18.31
8,TileStart,1822,18.22
9,OperatorCarConfirm,658,6.58
